In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
train_df=pd.read_csv("/kaggle/input/playground-series-s5e11/train.csv")
test_df=pd.read_csv("/kaggle/input/playground-series-s5e11/test.csv")

In [4]:
train_df.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [5]:
test_df

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,Female,Single,High School,Employed,Other,D5
1,593995,46626.39,0.093,732,15492.25,12.85,Female,Married,Master's,Employed,Other,C1
2,593996,54954.89,0.367,611,3796.41,13.29,Male,Single,Bachelor's,Employed,Debt consolidation,D1
3,593997,25644.63,0.110,671,6574.30,9.57,Female,Single,Bachelor's,Employed,Debt consolidation,C3
4,593998,25169.64,0.081,688,17696.89,12.80,Female,Married,PhD,Employed,Business,C1
...,...,...,...,...,...,...,...,...,...,...,...,...
254564,848558,92835.97,0.068,744,29704.00,13.48,Female,Single,Bachelor's,Employed,Debt consolidation,B2
254565,848559,48846.47,0.091,634,20284.33,9.58,Female,Married,High School,Employed,Debt consolidation,D4
254566,848560,20668.52,0.096,718,26387.55,9.00,Male,Single,Master's,Employed,Debt consolidation,C4
254567,848561,34105.09,0.094,739,11107.36,9.81,Male,Single,Bachelor's,Employed,Business,C2


In [6]:
train_df.describe()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,loan_paid_back
count,593994.000000,593994.000000,593994.000000,593994.000000,593994.000000,593994.000000,593994.000000
mean,296996.500000,48212.202976,0.120696,680.916009,15020.297629,12.356345,0.798820
std,171471.442235,26711.942078,0.068573,55.424956,6926.530568,2.008959,0.400883
min,0.000000,6002.430000,0.011000,395.000000,500.090000,3.200000,0.000000
25%,148498.250000,27934.400000,0.072000,646.000000,10279.620000,10.990000,1.000000
50%,296996.500000,46557.680000,0.096000,682.000000,15000.220000,12.370000,1.000000
75%,445494.750000,60981.320000,0.156000,719.000000,18858.580000,13.680000,1.000000
max,593993.000000,393381.740000,0.627000,849.000000,48959.950000,20.990000,1.000000


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 593994 entries, 0 to 593993
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    593994 non-null  int64  
 1   annual_income         593994 non-null  float64
 2   debt_to_income_ratio  593994 non-null  float64
 3   credit_score          593994 non-null  int64  
 4   loan_amount           593994 non-null  float64
 5   interest_rate         593994 non-null  float64
 6   gender                593994 non-null  object 
 7   marital_status        593994 non-null  object 
 8   education_level       593994 non-null  object 
 9   employment_status     593994 non-null  object 
 10  loan_purpose          593994 non-null  object 
 11  grade_subgrade        593994 non-null  object 
 12  loan_paid_back        593994 non-null  float64
dtypes: float64(5), int64(2), object(6)
memory usage: 58.9+ MB


In [8]:
print(f"the shape of the training data set is {train_df.shape}")
print(f"the shape of the test data set is {test_df.shape}")

the shape of the training data set is (593994, 13)
the shape of the test data set is (254569, 12)


In [9]:
print("Integer type data columns: id, annual_income,debt_to_income_ratio, credit score, loan_amount, interest_rate, loan_paid_back")
print("Categorical type data columns: gender, martial_status, education_level,employment_status, loan_purpose, grade_subgrade")
print("target column: loan_paid_back")

Integer type data columns: id, annual_income,debt_to_income_ratio, credit score, loan_amount, interest_rate, loan_paid_back
Categorical type data columns: gender, martial_status, education_level,employment_status, loan_purpose, grade_subgrade
target column: loan_paid_back


In [10]:
def create_features(df):
    df = df.copy()
    
    # Creating new features
    df['income_to_loan_ratio'] = df['annual_income'] / (df['loan_amount'] + 1)
    df['debt_burden'] = df['annual_income'] * df['debt_to_income_ratio']
    df['affordability_ratio'] = (df['annual_income'] / 12) / (df['loan_amount'] * df['interest_rate'] / 1200 + 1)
    df['credit_income_ratio'] = df['credit_score'] / df['annual_income']
    
    # Creating risk score based on multiple factors
    df['risk_score'] = (
        df['debt_to_income_ratio'] * 0.3 + 
        (800 - df['credit_score']) / 800 * 0.3 + 
        df['interest_rate'] / 25 * 0.2 +
        (df['loan_amount'] / df['annual_income']) * 0.2
    )
    
    # Extracting subgrade as numerical feature
    if 'grade_subgrade' in df.columns:
        df['grade'] = df['grade_subgrade'].str[0]
        df['subgrade_num'] = df['grade_subgrade'].str[1].astype(int)
    
    # Creating employment stability feature
    employment_mapping = {
        'Unemployed': 0,
        'Student': 1,
        'Self-employed': 2,
        'Employed': 3,
        'Retired': 2
    }
    df['employment_stability'] = df['employment_status'].map(employment_mapping)
    
    # Education level encoding
    education_mapping = {
        'High School': 1,
        'Other': 2,
        'Bachelor\'s': 3,
        'Master\'s': 4,
        'PhD': 5
    }
    df['education_num'] = df['education_level'].map(education_mapping)
    
    return df

# Applying feature engineering
train_df_eng = create_features(train_df)
test_df_eng = create_features(test_df)


In [11]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier
import warnings
warnings.filterwarnings('ignore')

In [12]:
def preprocess_data(train_df, test_df):
    # Selecting features for modeling
    feature_columns = [
        'annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate',
        'income_to_loan_ratio', 'debt_burden', 'affordability_ratio', 'credit_income_ratio', 
        'risk_score', 'employment_stability', 'education_num', 'subgrade_num'
    ]
    
    categorical_cols = ['gender', 'marital_status', 'loan_purpose', 'grade']
    
    # Combining train and test for consistent encoding
    combined = pd.concat([train_df, test_df], axis=0)
    
    # Labeling encode categorical variables
    label_encoders = {}
    for col in categorical_cols:
        if col in combined.columns:
            le = LabelEncoder()
            combined[col] = le.fit_transform(combined[col].astype(str))
            label_encoders[col] = le
    
    # Spliting back
    train_processed = combined.iloc[:len(train_df)].copy()
    test_processed = combined.iloc[len(train_df):].copy()
    
    # Preparing final feature set
    all_features = feature_columns + categorical_cols
    
    X_train = train_processed[all_features]
    y_train = train_processed['loan_paid_back']
    X_test = test_processed[all_features]
    
    # Scale numerical features
    scaler = StandardScaler()
    numerical_features_to_scale = [f for f in all_features if f not in categorical_cols]
    X_train[numerical_features_to_scale] = scaler.fit_transform(X_train[numerical_features_to_scale])
    X_test[numerical_features_to_scale] = scaler.transform(X_test[numerical_features_to_scale])
    
    return X_train, y_train, X_test, scaler, label_encoders

In [13]:
X_train, y_train, X_test, scaler, label_encoders = preprocess_data(train_df_eng, test_df_eng)

print("Training features shape:", X_train.shape)
print("Test features shape:", X_test.shape)
print("Features used:", X_train.columns.tolist())

Training features shape: (593994, 17)
Test features shape: (254569, 17)
Features used: ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'income_to_loan_ratio', 'debt_burden', 'affordability_ratio', 'credit_income_ratio', 'risk_score', 'employment_stability', 'education_num', 'subgrade_num', 'gender', 'marital_status', 'loan_purpose', 'grade']


In [15]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# Splitting data for validation
X = X_train
y = y_train
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {X_train_split.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

# Define parameter grids for each model
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 6, 10],
    'learning_rate': [0.01, 0.001, 0.1],
    'subsample': [0.5, 0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

cat_param_grid = {
    'iterations': [50, 100, 200],
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1]
}

rf_param_grid = {
    'n_estimators': [100],
    'max_depth': [8],
    'min_samples_split': [5],
    'min_samples_leaf': [2]
}

# Base models with default parameters for grid search
xgb_base = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

cat_base = CatBoostClassifier(
    random_state=42,
    verbose=False,
    thread_count=-1
)

rf_base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

print("\nPerforming grid search...")

# Create a progress bar for the overall process
with tqdm(total=4, desc="Overall Progress", position=0) as pbar_overall:
    
    # Grid search for XGBoost
    pbar_overall.set_description("Grid searching XGBoost")
    xgb_grid = GridSearchCV(xgb_base, xgb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
    xgb_grid.fit(X_train_split, y_train_split)
    xgb_model = xgb_grid.best_estimator_
    xgb_val_pred = xgb_model.predict_proba(X_val)[:, 1]
    print(f"✓ Best XGBoost parameters: {xgb_grid.best_params_}")
    print(f"  Best XGBoost CV score: {xgb_grid.best_score_:.4f}")
    pbar_overall.update(1)
    
    # Grid search for CatBoost
    pbar_overall.set_description("Grid searching CatBoost")
    cat_grid = GridSearchCV(cat_base, cat_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
    cat_grid.fit(X_train_split, y_train_split)
    cat_model = cat_grid.best_estimator_
    cat_val_pred = cat_model.predict_proba(X_val)[:, 1]
    print(f"✓ Best CatBoost parameters: {cat_grid.best_params_}")
    print(f"  Best CatBoost CV score: {cat_grid.best_score_:.4f}")
    pbar_overall.update(1)
    
    # Grid search for Random Forest
    pbar_overall.set_description("Grid searching Random Forest")
    rf_grid = GridSearchCV(rf_base, rf_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
    rf_grid.fit(X_train_split, y_train_split)
    rf_model = rf_grid.best_estimator_
    rf_val_pred = rf_model.predict_proba(X_val)[:, 1]
    print(f"✓ Best Random Forest parameters: {rf_grid.best_params_}")
    print(f"  Best Random Forest CV score: {rf_grid.best_score_:.4f}")
    pbar_overall.update(1)
    
    # Creating meta-features for stacking
    pbar_overall.set_description("Training meta-model")
    meta_features_val = np.column_stack([xgb_val_pred, cat_val_pred, rf_val_pred])
    
    # Grid search for meta-model (Logistic Regression)
    meta_param_grid = {
        'C': [0.1, 1.0, 10.0],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    }
    
    meta_base = LogisticRegression(random_state=42)
    meta_grid = GridSearchCV(meta_base, meta_param_grid, cv=3, scoring='roc_auc', verbose=0)
    meta_grid.fit(meta_features_val, y_val)
    meta_model = meta_grid.best_estimator_
    print(f"✓ Best meta-model parameters: {meta_grid.best_params_}")
    print(f"  Best meta-model CV score: {meta_grid.best_score_:.4f}")
    pbar_overall.update(1)

print("\n" + "="*50)
print("Meta-model trained successfully!")
print("="*50)

# Evaluating base models
print("\nBase Model Performance (Validation AUC):")
print(f"XGBoost:          {roc_auc_score(y_val, xgb_val_pred):.4f}")
print(f"CatBoost:         {roc_auc_score(y_val, cat_val_pred):.4f}")
print(f"Random Forest:    {roc_auc_score(y_val, rf_val_pred):.4f}")

# Ensemble performance
ensemble_val_pred = meta_model.predict_proba(meta_features_val)[:, 1]
print(f"\nStacking Ensemble: {roc_auc_score(y_val, ensemble_val_pred):.4f}")
print("="*50)

Training samples: 475195
Validation samples: 118799

Performing grid search...


Grid searching CatBoost:  25%|██▌       | 1/4 [24:04<1:12:13, 1444.62s/it]

✓ Best XGBoost parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1.0}
  Best XGBoost CV score: 0.9173


Grid searching Random Forest:  50%|█████     | 2/4 [29:34<26:17, 788.79s/it]

✓ Best CatBoost parameters: {'depth': 8, 'iterations': 200, 'learning_rate': 0.1}
  Best CatBoost CV score: 0.9165


Training meta-model:  75%|███████▌  | 3/4 [32:00<08:15, 495.43s/it]         

✓ Best Random Forest parameters: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
  Best Random Forest CV score: 0.9102


Training meta-model: 100%|██████████| 4/4 [32:11<00:00, 482.86s/it]

✓ Best meta-model parameters: {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'}
  Best meta-model CV score: 0.9176

Meta-model trained successfully!

Base Model Performance (Validation AUC):
XGBoost:          0.9174
CatBoost:         0.9164
Random Forest:    0.9097

Stacking Ensemble: 0.9176


In [16]:
# Retraining base models on full training data using best parameters
print("\nRetraining models on full training data with best parameters...")

# Create new models with best parameters
best_xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1,
    **xgb_grid.best_params_
)

best_cat = CatBoostClassifier(
    random_state=42,
    verbose=False,
    thread_count=-1,
    **cat_grid.best_params_
)

best_rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    **rf_grid.best_params_
)

print("Training XGBoost with best parameters...", end=" ")
best_xgb.fit(X_train, y_train)
print("✓")
print(f"Using parameters: {xgb_grid.best_params_}")

print("Training CatBoost with best parameters...", end=" ")
best_cat.fit(X_train, y_train)
print("✓")
print(f"Using parameters: {cat_grid.best_params_}")

print("Training Random Forest with best parameters...", end=" ")
best_rf.fit(X_train, y_train)
print("✓")
print(f"Using parameters: {rf_grid.best_params_}")

print("\nGenerating test predictions...")

xgb_test_pred = best_xgb.predict_proba(X_test)[:, 1]
cat_test_pred = best_cat.predict_proba(X_test)[:, 1]
rf_test_pred = best_rf.predict_proba(X_test)[:, 1]

# Creating meta-features for test set
meta_features_test = np.column_stack([xgb_test_pred, cat_test_pred, rf_test_pred])

# Create final meta-model with best parameters
best_meta = LogisticRegression(random_state=42, **meta_grid.best_params_)
best_meta.fit(meta_features_val, y_val)  # Train on validation data as before

# Final ensemble predictions
final_predictions = best_meta.predict_proba(meta_features_test)[:, 1]

print("All predictions generated using best parameters!")
print("\nFinal Model Parameters Used:")
print("XGBoost:", xgb_grid.best_params_)
print("CatBoost:", cat_grid.best_params_)
print("Random Forest:", rf_grid.best_params_)
print("Meta-model:", meta_grid.best_params_)


Retraining models on full training data with best parameters...
Training XGBoost with best parameters... ✓
Using parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1.0}
Training CatBoost with best parameters... ✓
Using parameters: {'depth': 8, 'iterations': 200, 'learning_rate': 0.1}
Training Random Forest with best parameters... ✓
Using parameters: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}

Generating test predictions...
All predictions generated using best parameters!

Final Model Parameters Used:
XGBoost: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 1.0}
CatBoost: {'depth': 8, 'iterations': 200, 'learning_rate': 0.1}
Random Forest: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Meta-model: {'C': 10.0, 'penalty': 'l2', 'solver': 'liblinear'}


In [17]:
# submission file
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'loan_paid_back': final_predictions,
})

print("Prediction Summary:")
print(f"Probability range: {final_predictions.min():.4f} - {final_predictions.max():.4f}")
print(f"Mean probability: {final_predictions.mean():.4f}")

submission_df.to_csv('submission.csv', index=False)

print(submission_df.head())

Prediction Summary:
Probability range: 0.0208 - 0.9641
Mean probability: 0.7997
       id  loan_paid_back
0  593994        0.944406
1  593995        0.957600
2  593996        0.290659
3  593997        0.944264
4  593998        0.951113
